# 10. Validation Summary

Aggregates synthetic validation results:
1. Model identification accuracy (GS and SBI, static and learning)
2. Confusion matrices — are errors symmetric?
3. Parameter recovery — do we recover the true parameters?
4. GS vs SBI agreement on synthetic ground truth

**Depends on:** Cluster jobs for synth_gs and synth_sbi.
Run `gather_cv_results.py --include-validation --all` first for GS summaries.

In [ ]:
%matplotlib inline
from shared_setup import *
import pickle
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec

In [ ]:
RESULTS_DIR = Path('../results/validation')
SYNTH_COHORTS_DIR = RESULTS_DIR / 'synthetic_cohorts'

COHORTS = ['static_uniform', 'learning_uniform']
FIT_TARGETS = ['update_matrix', 'conditional_psych']
FT_SHORT = {'update_matrix': 'UM', 'conditional_psych': 'CP'}

## 1. Load Results

Load both the gathered summaries (for accuracy) and the raw per-animal
pickles (for parameter recovery).

In [ ]:
# ── GS: load raw pickles for parameter recovery ─────────────────────────
synth_gs_raw = {}   # {(cohort, ft): list of raw pickle dicts}
synth_gs_dfs = {}   # {(cohort, ft): summary DataFrame}

for cohort in COHORTS:
    for ft in FIT_TARGETS:
        # Raw pickles (for parameter recovery)
        raw_dir = RESULTS_DIR / 'synth_gs' / f'{cohort}_{ft}'
        if raw_dir.exists():
            raw_files = sorted(raw_dir.glob('synth_*.pkl'))
            raws = []
            for p in raw_files:
                with open(p, 'rb') as f:
                    raws.append(pickle.load(f))
            if raws:
                synth_gs_raw[(cohort, ft)] = raws
                print(f'GS raw:  {cohort}/{ft} — {len(raws)} files')

        # Summary (from gather_cv_results.py)
        summary_path = raw_dir / 'summary.pkl'
        if summary_path.exists():
            with open(summary_path, 'rb') as f:
                synth_gs_dfs[(cohort, ft)] = pickle.load(f)['df']
            print(f'GS summ: {cohort}/{ft} — {len(synth_gs_dfs[(cohort, ft)])} animals')

# ── SBI: load raw pickles ─────────────────────────────────────────────────
synth_sbi_dfs = {}

for cohort in COHORTS:
    for ft in FIT_TARGETS:
        sbi_dir = RESULTS_DIR / 'synth_sbi' / f'{cohort}_{ft}'
        if not sbi_dir.exists():
            continue
        sbi_files = sorted(sbi_dir.glob('synth_*.pkl'))
        if sbi_files:
            rows = []
            for p in sbi_files:
                with open(p, 'rb') as f:
                    rows.append(pickle.load(f))
            synth_sbi_dfs[(cohort, ft)] = pd.DataFrame(rows)
            print(f'SBI:     {cohort}/{ft} — {len(rows)} files')

# ── Load cohort files (for true params) ───────────────────────────────────
cohort_data = {}
for cohort in COHORTS:
    path = SYNTH_COHORTS_DIR / f'{cohort}.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            cohort_data[cohort] = pickle.load(f)
        print(f'Cohort:  {cohort} — {len(cohort_data[cohort]["animals"])} animals')

## 2. Model Identification Accuracy

In [ ]:
def _build_accuracy_table():
    """Build a tidy accuracy table across all conditions."""
    rows = []
    for cohort in COHORTS:
        for ft in FIT_TARGETS:
            row = {'cohort': cohort, 'fit_target': ft}

            # GS
            gs_key = (cohort, ft)
            if gs_key in synth_gs_dfs:
                df = synth_gs_dfs[gs_key]
                col = 'correct' if 'correct' in df.columns else 'gs_correct'
                if col not in df.columns:
                    # Compute from winner vs true_model
                    if 'winner' in df.columns and 'true_model' in df.columns:
                        df['correct'] = df['winner'] == df['true_model']
                        col = 'correct'
                if col in df.columns:
                    row['gs_acc'] = df[col].mean()
                    row['gs_n'] = len(df)
                    row['gs_n_correct'] = int(df[col].sum())

            # SBI
            sbi_key = (cohort, ft)
            if sbi_key in synth_sbi_dfs:
                df = synth_sbi_dfs[sbi_key]
                col = 'correct' if 'correct' in df.columns else 'sbi_correct'
                if col in df.columns:
                    row['sbi_acc'] = df[col].mean()
                    row['sbi_n'] = len(df)
                    row['sbi_n_correct'] = int(df[col].sum())

            rows.append(row)
    return pd.DataFrame(rows)

acc_table = _build_accuracy_table()
print(acc_table.to_string(index=False, float_format='%.2f'))

In [ ]:
# ── Combined accuracy figure ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

colours = {'update_matrix': '#4C72B0', 'conditional_psych': '#DD8452'}
method_labels = {'gs_acc': 'Grid Search', 'sbi_acc': 'SBI (SNPE)'}

for ax_idx, (acc_col, method_label) in enumerate(method_labels.items()):
    ax = axes[ax_idx]
    sub = acc_table.dropna(subset=[acc_col])
    if len(sub) == 0:
        ax.text(0.5, 0.5, f'No {method_label} results',
                transform=ax.transAxes, ha='center', va='center')
        ax.set_title(method_label)
        continue

    x_labels = [f'{r["cohort"].replace("_", " ").title()}' for _, r in sub.iterrows()]
    x = np.arange(len(sub))
    w = 0.35

    for i, ft in enumerate(FIT_TARGETS):
        ft_sub = sub[sub['fit_target'] == ft]
        if len(ft_sub) == 0:
            continue
        ft_x = [j for j, (_, r) in enumerate(sub.iterrows()) if r['fit_target'] == ft]
        vals = ft_sub[acc_col].values
        n_col = acc_col.replace('acc', 'n_correct')
        n_total_col = acc_col.replace('acc', 'n')
        bars = ax.bar(
            np.array(ft_x) + (-0.18 + 0.36 * i), vals, 0.32,
            label=FT_SHORT[ft], color=colours[ft], alpha=0.8,
        )
        for bar, (_, r) in zip(bars, ft_sub.iterrows()):
            if n_col in r and n_total_col in r and pd.notna(r[n_col]):
                ax.text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f'{int(r[n_col])}/{int(r[n_total_col])}',
                    ha='center', va='bottom', fontsize=8,
                )

    ax.set_xticks(range(len(set(x_labels))))
    ax.set_xticklabels(sorted(set(x_labels)), fontsize=9)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8, label='Chance')
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Accuracy' if ax_idx == 0 else '')
    ax.set_title(method_label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')

fig.suptitle('Synthetic Validation: Model Identification Accuracy',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Confusion Matrices

Are misidentifications symmetric (equal chance of BE→SC and SC→BE errors)
or does one direction dominate?

In [ ]:
def _build_confusion(df, winner_col='winner', true_col='true_model'):
    """Build 2×2 confusion matrix from a results DataFrame."""
    if winner_col not in df.columns or true_col not in df.columns:
        return None
    labels = ['BE', 'SC']
    cm = np.zeros((2, 2), dtype=int)
    for _, r in df.iterrows():
        ti = labels.index(r[true_col]) if r[true_col] in labels else -1
        pi = labels.index(r[winner_col]) if r[winner_col] in labels else -1
        if ti >= 0 and pi >= 0:
            cm[ti, pi] += 1
    return cm


def _plot_confusion(cm, ax, title=''):
    labels = ['BE', 'SC']
    im = ax.imshow(cm, cmap='Blues', vmin=0)
    for i in range(2):
        for j in range(2):
            colour = 'white' if cm[i, j] > cm.max() * 0.6 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=14, fontweight='bold', color=colour)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(labels)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(labels)
    ax.set_xlabel('Assigned')
    ax.set_ylabel('True')
    total = cm.sum()
    acc = (cm[0, 0] + cm[1, 1]) / total if total > 0 else 0
    ax.set_title(f'{title}\n{acc:.0%} ({cm.trace()}/{total})', fontsize=10)


# Build confusion matrices for all available conditions
cms = {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        key = (cohort, ft)
        # GS
        if key in synth_gs_dfs:
            cm = _build_confusion(synth_gs_dfs[key])
            if cm is not None:
                cms[f'GS {FT_SHORT[ft]}\n{cohort.replace("_", " ")}'] = cm
        # SBI
        if key in synth_sbi_dfs:
            cm = _build_confusion(synth_sbi_dfs[key])
            if cm is not None:
                cms[f'SBI {FT_SHORT[ft]}\n{cohort.replace("_", " ")}'] = cm

if cms:
    n = len(cms)
    ncols = min(n, 4)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, (title, cm) in zip(axes, cms.items()):
        _plot_confusion(cm, ax, title)
    for i in range(n, len(axes)):
        axes[i].set_visible(False)
    fig.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No confusion data available.')

## 4. Parameter Recovery

For correctly identified animals, how well do the recovered parameters
match the true values? Uses the raw per-animal GS pickles which store
both `true_params` and `best_params_single`.

In [ ]:
def _extract_recovery_data(raw_pickles, cohort_animals):
    """
    Extract true vs recovered parameters from raw synth GS pickles.

    Strategy: for each synthetic animal, find the BE and SC fit pickles.
    Determine winner. If winner matches true_model, extract the winning
    model's best recovered params.

    Returns dict: {model_type: {param_name: {'true': [], 'recovered': []}}}
    """
    # Group raw pickles by animal_id
    by_animal = {}
    for r in raw_pickles:
        aid = r['animal_id']
        if aid not in by_animal:
            by_animal[aid] = {}
        by_animal[aid][r['model']] = r

    recovery = {'BE': {}, 'SC': {}}

    for aid, fits in by_animal.items():
        if 'BE' not in fits or 'SC' not in fits:
            continue

        be_err = fits['BE']['mean_error']
        sc_err = fits['SC']['mean_error']
        if np.isnan(be_err) or np.isnan(sc_err):
            continue

        winner = 'BE' if be_err < sc_err else 'SC'
        true_model = fits['BE']['true_model']  # same in both

        if winner != true_model:
            continue  # only plot recovery for correctly identified animals

        # Get true params
        true_params = fits[winner]['true_params']
        # true_params is a BEParams or SCParams dataclass
        if hasattr(true_params, '__dict__'):
            true_dict = {k: v for k, v in vars(true_params).items()
                         if not k.startswith('_') and isinstance(v, (int, float))}
        elif isinstance(true_params, dict):
            true_dict = true_params
        else:
            continue

        # Get recovered params from best seed
        seed_results = fits[winner].get('results', [])
        valid_results = [r for r in seed_results
                         if not np.isnan(r.get('avg_test_error', np.nan))
                         and 'best_params_single' in r]
        if not valid_results:
            continue

        best_idx = int(np.argmin([r['avg_test_error'] for r in valid_results]))
        recovered = valid_results[best_idx]['best_params_single']
        if not recovered:
            continue

        for pname, true_val in true_dict.items():
            if pname in recovered:
                if pname not in recovery[winner]:
                    recovery[winner][pname] = {'true': [], 'recovered': []}
                recovery[winner][pname]['true'].append(true_val)
                recovery[winner][pname]['recovered'].append(recovered[pname])

    return recovery


# Extract recovery data from static_uniform (cleanest scenario)
recovery_cohort = 'static_uniform'
recovery_ft = 'update_matrix'  # primary fit target
recovery_key = (recovery_cohort, recovery_ft)

if recovery_key in synth_gs_raw:
    recovery = _extract_recovery_data(
        synth_gs_raw[recovery_key],
        cohort_data.get(recovery_cohort, {}).get('animals', []),
    )
    for mt in ['BE', 'SC']:
        if recovery[mt]:
            n = len(next(iter(recovery[mt].values()))['true'])
            print(f'{mt}: {n} correctly identified animals with recovery data')
            for pname in recovery[mt]:
                t = np.array(recovery[mt][pname]['true'])
                r = np.array(recovery[mt][pname]['recovered'])
                corr = np.corrcoef(t, r)[0, 1] if len(t) > 2 else np.nan
                rmse = np.sqrt(np.mean((t - r) ** 2))
                print(f'  {pname:20s}  r={corr:.3f}  RMSE={rmse:.4f}')
else:
    recovery = {'BE': {}, 'SC': {}}
    print(f'No raw GS pickles for {recovery_cohort}/{recovery_ft}')

In [ ]:
# ── Parameter recovery scatter plots ──────────────────────────────────────
from plotting.recovery import plot_param_recovery_scatter

for mt in ['BE', 'SC']:
    params = recovery.get(mt, {})
    if not params:
        continue

    param_names = list(params.keys())
    n_params = len(param_names)
    if n_params == 0:
        continue

    ncols = min(n_params, 4)
    nrows = int(np.ceil(n_params / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    model_colour = '#4C72B0' if mt == 'BE' else '#DD8452'

    for i, pname in enumerate(param_names):
        true_arr = np.array(params[pname]['true'])
        rec_arr = np.array(params[pname]['recovered'])
        plot_param_recovery_scatter(
            true_arr, rec_arr, pname,
            ax=axes[i], color=model_colour,
        )

    for i in range(n_params, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle(
        f'{mt} Parameter Recovery — {recovery_cohort.replace("_", " ").title()}, '
        f'{FT_SHORT[recovery_ft]} (n={len(params[param_names[0]]["true"])} animals)',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()

## 5. GS vs SBI Agreement (Synthetic)

Do the two methods assign the same model to the same synthetic animal?

In [ ]:
from analysis.validation import summarise_agreement

for cohort in COHORTS:
    for ft in FIT_TARGETS:
        gs_key = (cohort, ft)
        sbi_key = (cohort, ft)
        if gs_key in synth_gs_dfs and sbi_key in synth_sbi_dfs:
            gs_df = synth_gs_dfs[gs_key].copy()
            sbi_df = synth_sbi_dfs[sbi_key].copy()

            # Normalise column names for merge
            if 'gs_winner' not in gs_df.columns and 'winner' in gs_df.columns:
                gs_df = gs_df.rename(columns={'winner': 'gs_winner'})
            if 'gs_correct' not in gs_df.columns and 'correct' in gs_df.columns:
                gs_df = gs_df.rename(columns={'correct': 'gs_correct'})
            if 'sbi_winner' not in sbi_df.columns and 'winner' in sbi_df.columns:
                sbi_df = sbi_df.rename(columns={'winner': 'sbi_winner'})
            if 'sbi_correct' not in sbi_df.columns and 'correct' in sbi_df.columns:
                sbi_df = sbi_df.rename(columns={'correct': 'sbi_correct'})

            summarise_agreement(
                gs_df, sbi_df,
                label=f'{cohort} / {ft}',
            )

## 6. Summary

Key numbers for the presentation:
- GS model ID accuracy (static / learning × UM / CP)
- SBI model ID accuracy (same breakdown)
- Parameter recovery correlation (per model, per parameter)
- Error symmetry from confusion matrices
- GS–SBI agreement rate